In [4]:
import math

import numpy as np
import pandas as pd
import xgboost as xgb
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

In [5]:
INDEX_TICKER = "^NSEI"
# CONSTITUENTS = {
#     "HDFCBANK": "HDFCBANK.NS",
#     "RELIANCE": "RELIANCE.NS",
#     "ICICIBANK": "ICICIBANK.NS",
#     "INFY": "INFY.NS",
#     "TCS": "TCS.NS",
# }

# # Approximate free-float index weights as fractions of the FULL index. Do not normalize.
# # Verify against the current NSE factsheet before the final run.
# INDEX_WEIGHTS = {
#     "HDFCBANK": 0.115,
#     "RELIANCE": 0.095,
#     "ICICIBANK": 0.080,
#     "INFY": 0.055,
#     "TCS": 0.040,
# }


CONSTITUENTS = {
    "HDFCBANK": "HDFCBANK.NS",
    "ICICIBANK": "ICICIBANK.NS",
    "RELIANCE": "RELIANCE.NS",
    "BHARTIARTL": "BHARTIARTL.NS",
    "LT": "LT.NS",
}

INDEX_WEIGHTS = {
    "HDFCBANK": 0.1054,
    "ICICIBANK": 0.0881,
    "RELIANCE": 0.0752,
    "BHARTIARTL": 0.0512,
    "LT": 0.0417,
}

REST_WEIGHT = 1 - sum(INDEX_WEIGHTS.values())   # share of index not modelled bottom-up
REST_PRED_RETURN = 0.0                          # ASSUMPTION for the rest; state it in the paper

START = "2018-01-01"
SPLIT_DATE = pd.Timestamp("2024-01-01")
FEATURES = ["Return", "SMA_10", "SMA_50", "RSI", "MACD", "Volume"]

In [6]:
def download(ticker):
    df = yf.download(ticker, start=START, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Close", "Volume"]]

raw = {name: download(t) for name, t in CONSTITUENTS.items()}
raw["NIFTY"] = download(INDEX_TICKER)

/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

In [7]:
close = pd.concat({k: v["Close"] for k, v in raw.items()}, axis=1, join="inner").dropna()
volume = pd.concat({k: v["Volume"] for k, v in raw.items()}, axis=1).loc[close.index]
print(f"Common trading days: {len(close)} ({close.index.min().date()} to {close.index.max().date()})")

Common trading days: 2186 (2018-01-02 to 2026-11-16)


In [8]:
def build_features(symbol):
    c = close[symbol]
    d = pd.DataFrame(index=close.index)
    d["Return"] = c.pct_change()
    d["SMA_10"] = c.rolling(10).mean()
    d["SMA_50"] = c.rolling(50).mean()
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    d["RSI"] = 100 - 100 / (1 + gain / (loss + 1e-9))
    d["MACD"] = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    d["Volume"] = volume[symbol]
    # Target: next-day return. Features at t use only data up to t.
    d["Target_Return"] = d["Return"].shift(-1)
    return d.dropna()

feats = {s: build_features(s) for s in close.columns}
common = feats["NIFTY"].index
for s in feats:
    common = common.intersection(feats[s].index)
feats = {s: f.loc[common] for s, f in feats.items()}

In [9]:
def fit_predict(train_df, test_df):
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(train_df[FEATURES])
    X_test = scaler.transform(test_df[FEATURES])
    model = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
    model.fit(X_train, train_df["Target_Return"])
    return pd.Series(model.predict(X_test), index=test_df.index)

pred_ret = {}
for s in list(CONSTITUENTS) + ["NIFTY"]:
    d = feats[s]
    pred_ret[s] = fit_predict(d[d.index < SPLIT_DATE], d[d.index >= SPLIT_DATE])
pred_ret = pd.DataFrame(pred_ret)
test_idx = pred_ret.index
print(f"Test days: {len(test_idx)} (from {test_idx.min().date()})")

Test days: 708 (from 2024-01-01)


In [10]:
nifty_close = close["NIFTY"].loc[test_idx]
actual_next = close["NIFTY"].shift(-1).loc[test_idx]   # Nifty close on the next trading day
mask = actual_next.notna()

R_A = pred_ret["NIFTY"]
R_B = sum(INDEX_WEIGHTS[s] * pred_ret[s] for s in CONSTITUENTS) + REST_WEIGHT * REST_PRED_RETURN

pred_A = nifty_close * (1 + R_A)     # Model A: top-down
pred_B = nifty_close * (1 + R_B)     # Model B: bottom-up top 5 (+ rest assumption)
naive = nifty_close                  # persistence benchmark: tomorrow = today

In [11]:
a = actual_next[mask]
c = nifty_close[mask]
up_actual = (a > c).values

def point_metrics(pred, name):
    p = pred[mask]
    return {
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(a, p)),
        "MAPE_%": mean_absolute_percentage_error(a, p) * 100,
    }

def mda(pred):
    return ((pred[mask] > c).values == up_actual).mean() * 100

results = pd.DataFrame([
    point_metrics(pred_A, "Model A (Top-Down)"),
    point_metrics(pred_B, "Model B (Bottom-Up Top 5)"),
    point_metrics(naive, "Naive (persistence)"),
]).set_index("Model")
results["MDA_%"] = [mda(pred_A), mda(pred_B), np.nan]

print(results.round(2).to_string())
print(f"\nAlways-predict-UP hit rate (base rate): {up_actual.mean() * 100:.2f}%")

                             RMSE  MAPE_%  MDA_%
Model                                           
Model A (Top-Down)         228.36    0.71  52.26
Model B (Bottom-Up Top 5)  208.68    0.64  46.61
Naive (persistence)        203.59    0.62    NaN

Always-predict-UP hit rate (base rate): 51.69%


In [12]:
def diebold_mariano(pred1, pred2):
    e1 = (a - pred1[mask]).values
    e2 = (a - pred2[mask]).values
    d = e1 ** 2 - e2 ** 2                 # d > 0 means Model 1 has larger loss
    dm = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
    p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(dm) / math.sqrt(2))))
    return dm, p_value

dm_stat, dm_p = diebold_mariano(pred_B, pred_A)
print(f"Diebold-Mariano (B vs A, squared error): DM = {dm_stat:.3f}, p = {dm_p:.4f}")
print("Positive DM means Model B has the larger loss. Compare against the naive benchmark as well.")

Diebold-Mariano (B vs A, squared error): DM = -4.379, p = 0.0000
Positive DM means Model B has the larger loss. Compare against the naive benchmark as well.


In [13]:
out = pd.DataFrame({
    "Actual": a,
    "Naive_Pred": naive[mask],
    "TopDown_Pred": pred_A[mask],
    "BottomUp_Pred": pred_B[mask],
})
out.index.name = "Date"
out.to_csv("comparison_results_v2.csv")
print("Saved comparison_results_v2.csv")

Saved comparison_results_v2.csv
